<a href="https://colab.research.google.com/github/hanokjoshua144/DEEP-LEARNING-6-SEM/blob/main/GUIBP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class SimplePerceptron(nn.Module):
    def __init__(self):
        super(SimplePerceptron, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

Modify ReLU for Guided Backprop

In [ ]:
class GuidedReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors

        grad_input = grad_output.clone()

        # Step 1: Block negative gradients
        grad_input[grad_output < 0] = 0

        # Step 2: Block neurons with negative activation
        grad_input[input < 0] = 0

        return grad_input

Replace ReLU with GuidedReLU

In [ ]:
class GuidedModel(nn.Module):
    def __init__(self, model):
        super(GuidedModel, self).__init__()
        self.model = model

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.model.fc1(x)
        x = GuidedReLU.apply(x)   # 👈 replaced
        x = self.model.fc2(x)
        return x

Prepare Input Image

In [ ]:
input_image = torch.randn(1, 1, 28, 28, requires_grad=True)

Forward Pass

In [ ]:
model = SimplePerceptron()
guided_model = GuidedModel(model)

output = guided_model(input_image)

Select Target Class

In [ ]:
target_class = output.argmax()

one_hot = torch.zeros_like(output)
one_hot[0][target_class] = 1

Backward Pass (Core Step 🔥)

In [ ]:
guided_model.zero_grad()
output.backward(gradient=one_hot)

Extract Gradients (Important Pixels)

In [ ]:
guided_gradients = input_image.grad.data.numpy()[0][0]

Visualize

In [ ]:
plt.imshow(guided_gradients, cmap='hot')
plt.title("Guided Backpropagation")
plt.colorbar()
plt.show()

Observations

1. Sparse Activation Map
Only important pixels are highlighted
Noise is reduced
2. High Interpretability
Clear regions influencing prediction
Better than normal gradients
3. Positive Influence Only
Shows pixels that support the class
Ignores negative contributions
4. Sharper than Vanilla Backprop
Edges and shapes become clearer
5. Model Insight
Helps understand:
What features model learned
Whether model focuses on correct region